**Model script for the paper 'Automatic Detection of Immediate and Self Repetitions in Naturalistic Speech Recordings of French- and Dutch-Speaking Autistic Children'**

This script contains the NLP model described in the paper 'Automatic Detection of Immediate and Self Repetitions in Naturalistic Speech Recordings of French- and Dutch-Speaking Autistic Children'. The model aims at detecting repetitions in the speech of a target child, more specifically direct repetitions from utterances of other speakers pronounced less than 10 seconds before the target utterance, or self-repetitions (from utterances spoken by the target child inside the same sound file). For more information about the goals and the working of the model, see the paper. This model computes linguistic similarity scores to distinguish repetitive from non-repetitive utterance pairs. The prediction is based on comparing the similarity score with a threshold determined in gold standard annotated data. Our thresholds are specific for Dutch and French, and using the default thresholds will thus only work for data in these languages. The user can specify own thresholds for use of the model on other languages. For a reflexion on the cross-linguistic applicability of the model, see the paper.

In [ ]:
class similarity:
    """ A class for computing similarity scores between linguistic segments based on a pre-trained BERT model.
    This class processes linguistic data, applies BERT embeddings, and calculates similarity metrics
    between specified linguistic levels (lexical, syntactic and/or semantic). 
    If thresholds for the similarity metrics are given, 
    the model produces a prediction ('non-repetitive' or 'repetitive') for each similarity measure by compairing
    the measure to the threshold.
    For the Dutch and French languages, if thresholds are not specified, the thresholds found to perform best in the
    paper are used by default.
    The output of the main functions 'get_predictions_direct_repetition' and 'get_predictions_self_repetition'
    are dictionaries containing the above described elements in a dictionary per utterance pair.
    """ 
    
    from sentence_transformers import SentenceTransformer, util
    import pickle
    import pandas as pd
    from praatio import textgrid
    import numpy as np
    import tgt
    import re
    from glob import glob
    import spacy
    from typing import List, Optional
    from itertools import product
  
    def __init__(
        self,
        textgrid_file: str,
        sbert_model: Optional[str] = None,
        spacy_model: Optional[str]= None,
        language: str= None,
        child_tier: str= None,
        non_speech_tiers: Optional[List[str]]= None
    
    ):

        
        """Initializes the similarity class with linguistic data and pre-trained models.
        
        Args:
            textgrid_file (str): Path to the TextGrid file containing speech intervals.
            sbert_model (Optional[str]): Path or name of Sentence-BERT model.
            spacy_model (Optional[str]): Path or name of SpaCy model.
            language (str): Language of the input data (language code as employed in SpaCy.
            child_tier (str): Tier name for speech intervals of the target child.
            non_speech_tiers (Optional[List[str]]): Tier names for non-speech intervals to exclude.
        """
   
    
        self.textgrid_file= textgrid_file
        self.child_tier= child_tier
        self.non_speech_tiers= non_speech_tiers
        
        default_sbert_models= {'fr': SentenceTransformer('Lajavaness/sentence-camembert-large'),
                       'nl': SentenceTransformer('NetherlandsForensicInstitute/robbert-2022-dutch-sentence-transformers')}
        
        self.language= language
        
        if not sbert_model:
            if language in ['nl','fr']:
                self.sbert_model= default_sbert_models[language]
            else:
                raise ValueError('Please specify the name of the SentenceBERT model you want to use')
        else:
            self.sbert_model= sbert_model
 
        self.spacy_model= spacy_model if spacy_model else language + '_core_news_sm'
        self.nlp= spacy.load(self.spacy_model)
    

        self.input_tg= textgrid.openTextgrid(self.textgrid_file, includeEmptyIntervals= True)
        
        self.child_intervals, self.other_speaker_intervals= self.get_speaker_dictionaries()
        self.child_sem_vectors, self.other_speaker_sem_vectors= self.get_semantic_vectors()
        self.child_lemmas, self.child_lexical_vectors, self.child_counts, self.other_speaker_lemmas, self.other_speaker_lexical_vectors, self.other_speaker_counts = self.get_lexical_vectors()
        
        
    def get_speech_intervals(self, tier, empty=False, filter_unintelligible='^(xxx|yyy)\s?(\[.+\])?\.?$'):
        """Takes a tier name as input and returns either the timestamps of the non-empty intervals ('empty= False')
        or those of the empty intervals ('empty=True') of that tier,
        filtering out unintelligible utterances if filtering regex completed"""
        entries= self.input_tg.getTier(tier).entries
        intervals= {}
        for entry in entries:
            if  (entry.label and empty==False and not filter_unintelligible) or (not entry.label and empty==True)\
                or (entry.label and empty==False and filter_unintelligible and not re.match(filter_unintelligible, entry.label)):
                intervals[(entry.start, entry.end)]= entry.label.translate(str.maketrans('','', '+?!,/.()[]')) # Remove punctuation
        return intervals
    
    def get_speaker_dictionaries(self):
        """Builds two speaker dictionaries: (i) for the autistic child and (ii) for all other speakers,
        using the function `get_speech_intervals`. The second dictionary contains an additional hierarchical level,
        i.e. the tier name of the speaker.
        """
        
        child_intervals= {}
        other_speaker_intervals= {}
        
        all_tiers= self.input_tg.tierNames
        all_intervals= {}
        for tier in all_tiers:
            if tier== self.child_tier:
                child_intervals= self.get_speech_intervals(tier)
            elif tier not in self.non_speech_tiers:
                other_speaker_intervals[tier]= self.get_speech_intervals(tier)

        
        return child_intervals, other_speaker_intervals
    
    def create_vectors_from_ling_unit(self,units_dict, all_units):
        """Creates numerical vectors from a dictionary containing linguistic units as value (e.g. lemma, POS n-gram),
        given a list of all units occurring in the entire file"""
        
        # Identify all unique units across all intervals + sort for consistent dimension ordering
        unique_units= sorted(set(all_units))
        unit_index_mapping= {unit: idx for idx, unit in enumerate(unique_units)}

        # Create vectors
        vectors = {
            interval: np.bincount(
                [unit_index_mapping[unit] for unit in units if unit in unit_index_mapping],
                minlength=len(unique_units)
            )
            for interval, units in units_dict.items()
        }
        return vectors
    
    def get_lexical_vectors(self):
        """Converts the speaker dictionaries into dictionaries of lemma vectors using spaCy.
        Also outputs lemma counts for each utterance that can be used for length normalisation in similarity comparison.
        """
        # Extract lemmas as before
        child_lemmas = {
            (start, end): np.array([token.lemma_ for token in self.nlp(speech)])
            for (start, end), speech in self.child_intervals.items()
        }

        other_speaker_lemmas = {
            speaker: {
                (start, end): np.array([token.lemma_ for token in self.nlp(speech)])
                for (start, end), speech in value.items()
            }
            for speaker, value in self.other_speaker_intervals.items()
        }

        all_lemmas = [lemma for lemmas in child_lemmas.values() for lemma in lemmas]
        all_lemmas += [
            lemma
            for speaker, intervals in other_speaker_lemmas.items()
            for lemmas in intervals.values()
            for lemma in lemmas]


        # Apply to child and other speakers
        child_lexical_vectors = self.create_vectors_from_ling_unit(child_lemmas, all_lemmas)

        other_speaker_lexical_vectors = {
            speaker: self.create_vectors_from_ling_unit(value, all_lemmas)
            for speaker, value in other_speaker_lemmas.items()
        }

        # Count lemmas as before
        child_counts = {
            (start, end): len(lemmas) for (start, end), lemmas in child_lemmas.items()
        }

        other_speaker_counts = {
            speaker: {
                (start, end): len(lemmas)
                for (start, end), lemmas in value.items()
            }
            for speaker, value in other_speaker_lemmas.items()
        }


        return child_lemmas, child_lexical_vectors, child_counts, other_speaker_lemmas, other_speaker_lexical_vectors, other_speaker_counts
    
    def get_syntactic_vectors(self, n=2):
        """
        Converts the speaker dictionaries into dictionaries of POS n-gram vectors using spaCy.
        Each unique n-gram constitutes a dimension of the vector, and the number of occurrences
        of that n-gram in a given utterance is the value under that dimension.
        """
    
        def extract_POS_ngrams(speech, n):
            # Extract n-grams of POS tags from the speech
            tokens = [token.pos_ for token in self.nlp(speech)]  # Use token.pos_ for string representation
            ngrams = [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]
            if len(tokens) < n:
                ngrams = [tuple(tokens)]
            return ngrams

        # Extract n-grams for child utterances
        child_ngrams = {
            (start, end): extract_POS_ngrams(speech, n)
            for (start, end), speech in self.child_intervals.items()
        }

        # Extract n-grams for other speakers
        other_speaker_ngrams = {
            speaker: {
                (start, end): extract_POS_ngrams(speech, n)
                for (start, end), speech in value.items()
            }
            for speaker, value in self.other_speaker_intervals.items()
        }


        all_ngrams= [ngram for ngrams in child_ngrams.values() for ngram in ngrams]
        all_ngrams += [
            ngram
            for speaker, intervals in other_speaker_ngrams.items()
            for ngrams in intervals.values()
            for ngram in ngrams]



        # Create n-gram vectors for child utterances
        child_syntactic_vectors = self.create_vectors_from_ling_unit(child_ngrams, all_ngrams)

        # Create n-gram vectors for other speakers
        other_speaker_syntactic_vectors = {
            speaker: self.create_vectors_from_ling_unit(value, all_ngrams)
            for speaker, value in other_speaker_ngrams.items()
        }

        return child_syntactic_vectors, other_speaker_syntactic_vectors, child_ngrams, other_speaker_ngrams
        
    def get_semantic_vectors(self):
        """Converts the speaker dictionaries into dictionaries of semantic vectors
        using SentenceTransformer models
        """
        
        child_vectors= {(start, end): self.sbert_model.encode([speech]) for (start,end), speech in self.child_intervals.items()}
        other_speaker_vectors= {speaker: {(start, end): self.sbert_model.encode([speech]) for (start,end), speech in value.items()} for speaker, value in self.other_speaker_intervals.items()}
        
        return child_vectors, other_speaker_vectors
    
    def correct_data_type(self,v1,v2):
        "Ensures correct data type for vectors for passing to sentence_transformers.utils.cos_sim"
        
        if v1.dtype != float:
            v1= v1.astype(np.float64)
        if v2.dtype != float:
            v2= v2.astype(np.float64)

        return v1, v2



    def pad_or_concatenate_vectors(self,vectors):
        """
        Pads or concatenates vectors of different lengths.
        Returns a single combined vector.
        """
        # Find the maximum dimension among all vectors for padding
        max_len = max(vectors, key=len).shape[0]
        padded_vectors = []

        for vector in vectors:
            # Pad each vector that is shorter
            if np.ndim(vector) > 1:
                vector= vector.flatten()
            if vector.shape[0] < max_len:
                padding = np.zeros(max_len - vector.shape[0])
                padded_vectors.append(np.concatenate([vector, padding]))
            else:
                padded_vectors.append(vector)

        # Concatenate all vectors into a single vector
        return np.concatenate(padded_vectors)


    def get_predictions_direct_repetition(self,
                                  time_distance: int = 10,
                                  sim_thresholds: Optional[List[float]] = None,
                                  linguistic_levels: List[str] = ['lexical', 'syntactic', 'semantic'],
                                  n_POS: Optional[int] = 2):
        
        
        """
        For each utterance pair suitable for being candidate for direct repetition,
        computes cosine similarity on the created lexical, syntactic and/or semantic vectors (as indicated in 'linguistic_levels'),
        compares them with given sim_thresholds if given, or our default thresholds for Dutch and French 
        if one of these is selected as language and no sim_thresholds are given,
        and outputs a dictionary containing the following for each utterance pair:
        - language, input textgrid filepath
        - utterance start and end timestamps
        - tier name of comparison speaker
        - utterance transcriptions
        - utterance length (in words)
        - the chosen cosine similarity metrics
        - one prediction per metric, if thresholds provided.
        
        
        Args:
            time_distance (int): Maximum distance in seconds between utterance of target child and source candidate (default: 10 seconds).
            sim_threholds (Optional[List[float]]): List of float values representing thresholds for each of the similarity values chosen under 'linguistic_levels'.
                If not provided and language is Dutch or French, uses thresholds found in the paper.
            linguistic_levels (List[str]): List of linguistic vectors (among 'lexical','syntactic','semantic') to use in similarity computation.
            n_POS (Optional[int]): n value for Part of Speech n-grams used to construct syntactic vectors (default= 2).
        
        
        """

        allowed_levels = {'lexical', 'syntactic', 'semantic'}
        
        default_thresholds= {'nl': {'lexical': 0.232, 'syntactic': 0.212, 'semantic':0.374},
                            'fr': {'lexical': 0.292, 'syntactic': 0.232, 'semantic': 0.394}}

        # Validate linguistic_levels
        if not isinstance(linguistic_levels, list) or not all(isinstance(level, str) for level in linguistic_levels):
            raise ValueError("linguistic_levels must be a list of strings.")
        if any(level not in allowed_levels for level in linguistic_levels):
            raise ValueError(f"linguistic_levels can only contain values from {allowed_levels}.")
            
        if not sim_thresholds:
            if self.language in ['nl', 'fr']:
                sim_thresholds= [default_thresholds[self.language][linguistic_level] for linguistic_level in linguistic_levels]
            else:
                raise ValueError('For languages other than Dutch and French, please insert a threshold list with one float value per linguistic level')



        sim_names = [f'{linguistic_level}_cos_sim' for linguistic_level in linguistic_levels]
        pred_names = [f'{linguistic_level}_prediction' for linguistic_level in linguistic_levels]

        # Get syntactic vectors for given n value
        self.child_syntactic_vectors, self.other_speaker_syntactic_vectors, self.child_ngrams, self.other_speaker_ngrams = self.get_syntactic_vectors(n=n_POS)

        # Vector dictionaries for comparison
        possible_vector_dicts = {'child': {'lexical': self.child_lexical_vectors,
                                           'syntactic': self.child_syntactic_vectors,
                                           'semantic': self.child_sem_vectors},
                                 'other_speakers': {'lexical': self.other_speaker_lexical_vectors,
                                                    'syntactic': self.other_speaker_syntactic_vectors,
                                                    'semantic': self.other_speaker_sem_vectors}}

        # Filter chosen vectors based on linguistic levels
        chosen_vector_dicts = {speaker: {level: dictionary for level, dictionary in possible_vector_dicts[speaker].items()
                                         if level in linguistic_levels}
                               for speaker in possible_vector_dicts}

        # Create direct repetition dictionary
        direct_rep_dict = {}

        keys = ['language', 'file', 's2_tier', 's2_interval', 'child_interval', 's2_speech', 'child_speech',
                's2_length', 'child_length'] + sim_names + pred_names


        i = 0
        # Comparison between child and other speaker
        for s2 in chosen_vector_dicts['other_speakers'][linguistic_levels[0]]:
            for (start_child, end_child), child_vector in chosen_vector_dicts['child'][linguistic_levels[0]].items():
                for (start_s2, end_s2), s2_vector in chosen_vector_dicts['other_speakers'][linguistic_levels[0]][s2].items():
                    if 0 < start_child - start_s2 <= time_distance:
                        

                        # Calculate cosine similarity
                        sim_measures = []
                        preds= []
                        
                        for index, linguistic_level in enumerate(linguistic_levels):

                            child_vector= chosen_vector_dicts['child'][linguistic_level][start_child, end_child]
                            s2_vector= chosen_vector_dicts['other_speakers'][linguistic_level][s2][start_s2, end_s2]
                            child_vector, s2_vector= self.correct_data_type(child_vector, s2_vector)
                                                       

                            child_length = self.child_counts[start_child, end_child]
                            s2_length = self.other_speaker_counts[s2][start_s2, end_s2]

                            cos_sim = float(util.cos_sim(child_vector, s2_vector))
                            sim_measures.append(cos_sim)
                            
                            pred= 'repetitive' if cos_sim > sim_thresholds[index] else 'non-repetitive'
                            preds.append(pred)
                        
                        


                        values = [self.language, self.textgrid_file, s2, (start_s2, end_s2), (start_child, end_child),
                                  self.other_speaker_intervals[s2][start_s2, end_s2], self.child_intervals[start_child, end_child],
                                  s2_length, child_length] + sim_measures + preds


                        direct_rep_dict[i] = {key: value for key, value in zip(keys, values)}
                        i += 1

                        
        return direct_rep_dict


    def get_predictions_self_repetition(self,
                                    sim_thresholds: Optional[List[float]] = None,
                                    linguistic_levels: List[str] = ['lexical', 'syntactic', 'semantic'],
                                    n_POS: Optional[int] = 2):
        
        
        """
        For each utterance pair suitable for being candidate for self-repetition,
        computes cosine similarity on the created lexical, syntactic and/or semantic vectors (as indicated in 'linguistic_levels'),
        compares them with given sim_thresholds if given, or our default thresholds for Dutch and French 
        if one of these is selected as language and no sim_thresholds are given,
        and outputs a dictionary containing the following for each utterance pair:
        - language, input textgrid filepath
        - utterance start and end timestamps
        - utterance transcriptions
        - utterance length (in words)
        - the chosen cosine similarity metrics
        - one prediction per metric, if thresholds provided.
        
        
        Args:
            time_distance (int): Maximum distance in seconds between utterance of target child and source candidate (default: 10 seconds).
            sim_threholds (Optional[List[float]]): List of float values representing thresholds for each of the similarity values chosen under 'linguistic_levels'.
                If not provided and language is Dutch or French, uses thresholds found in the paper.
            linguistic_levels (List[str]): List of linguistic vectors (among 'lexical','syntactic','semantic') to use in similarity computation.
            n_POS (Optional[int]): n value for Part of Speech n-grams used to construct syntactic vectors (default= 2).
        """
        
        allowed_levels = {'lexical', 'syntactic', 'semantic'}
        
        default_thresholds= {'nl': {'lexical': 0.919, 'syntactic': 0.879 , 'semantic': 0.879},
                            'fr': {'lexical': 0.879, 'syntactic': 0.899, 'semantic': 0.879}}

        # Validate linguistic_levels
        if not isinstance(linguistic_levels, list) or not all(isinstance(level, str) for level in linguistic_levels):
            raise ValueError("linguistic_levels must be a list of strings.")
        if any(level not in allowed_levels for level in linguistic_levels):
            raise ValueError(f"linguistic_levels can only contain values from {allowed_levels}.")
            
             
        if not sim_thresholds:
            if self.language in ['nl', 'fr']:
                sim_thresholds= [default_thresholds[self.language][linguistic_level] for linguistic_level in linguistic_levels]
            else:
                raise ValueError('For languages other than Dutch and French, please insert a threshold list with one float value per linguistic level')

        
        sim_names = [f'{linguistic_level}_cos_sim' for linguistic_level in linguistic_levels]
        pred_names = [f'{linguistic_level}_prediction' for linguistic_level in linguistic_levels]


        # Get correct syntactic vectors for chosen n_POS value
        self.child_syntactic_vectors, self.other_speaker_syntactic_vectors, self.child_ngrams, self.other_speaker_ngrams = self.get_syntactic_vectors(n=n_POS)

        # Select vector dictionaries that will be used to compare utterances, based on given linguistic comparison level
        possible_vector_dicts= {'lexical': self.child_lexical_vectors, 
                                'syntactic': self.child_syntactic_vectors, 
                                'semantic': self.child_sem_vectors}
                       
        chosen_vector_dicts = {level: dictionary for level, dictionary in possible_vector_dicts.items() if level in linguistic_levels}
        

        # Create self-repetition dictionary
        selfrep_dict = {}

        keys = ['language', 'file', 'source_interval', 'rep_interval', 'source_speech', 'rep_speech',
                'source_length', 'rep_length'] + sim_names + pred_names


        i = 0
        # Comparison between source and repetition for self-repetition
        for (start_source, end_source), source_vector in chosen_vector_dicts[linguistic_levels[0]].items():
            for (start_rep, end_rep), rep_vector in chosen_vector_dicts[linguistic_levels[0]].items():
                if start_source < start_rep:

                    # Calculate cosine similarity
                    sim_measures = []
                    preds= []
                    
                    for index, linguistic_level in enumerate(linguistic_levels):
                        source_vector= chosen_vector_dicts[linguistic_level][start_source, end_source]
                        rep_vector= chosen_vector_dicts[linguistic_level][start_rep, end_rep]
                        source_vector, rep_vector = self.correct_data_type(source_vector, rep_vector)

                        source_length= self.child_counts[start_source, end_source]
                        rep_length= self.child_counts[start_rep, end_rep]

                        cos_sim= float(util.cos_sim(source_vector, rep_vector)) 

                        sim_measures.append(cos_sim)
                        
                        pred= 'repetitive' if cos_sim > sim_thresholds[index] else 'non-repetitive'
                        preds.append(pred)
                    
                    


                    values = [self.language, self.textgrid_file, (start_source, end_source), (start_rep, end_rep),
                              self.child_intervals[start_source, end_source], self.child_intervals[start_rep, end_rep],
                              source_length, rep_length] + sim_measures + preds



                    selfrep_dict[i] = {key: value for key, value in zip(keys, values)}
                    i += 1

        return selfrep_dict

In [ ]:
# Example use
textgrid_file= "Your_transcription.textgrid"

test_model = similarity(sbert_model= None, spacy_model=None, language= 'nl', textgrid_file= textgrid_file, 
                        child_tier='Transcription', non_speech_tiers= ['other_tier1','other_tier2'])
    
selfrep_dict= test_model.get_predictions_self_repetition(
    sim_thresholds= None,
    linguistic_levels=  ['lexical', 'syntactic', 'semantic'], 
    n_POS= 2)

direct_rep_dict= test_model.get_predictions_direct_repetition(
    sim_thresholds= None,
    linguistic_levels=  ['lexical', 'syntactic', 'semantic'], 
    n_POS= 2)